# Section 9: End-to-End Traceability (Q77–85)

Full forward/backward tracing, procure-to-pay, order-to-cash, and raw-to-shelf costing.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import (
    get_session, run_sql, explode_bom, bom_to_df,
    build_transport_graph, resolve_location_key, display_path
)
import networkx as nx
import pandas as pd
conn, ontology = get_session()

## Q77

Starting from batch B-001-000021, trace forward to every retail location that received product from this batch — through shipments and their delivery destinations. Note that some shipment records may have missing foreign keys, so handle those gaps gracefully.

In [ ]:
# Forward trace: batch -> SKU -> shipment_lines -> shipments -> destination (retail)
run_sql(conn, """
    SELECT b.batch_number,
           s.sku_code,
           sh.shipment_number, sh.status as ship_status,
           sh.route_type, sh.destination_id,
           rl.location_code as retail_location, rl.city
    FROM batches b
    JOIN skus s ON b.product_id = s.id AND b.product_type = 'finished_good'
    LEFT JOIN shipment_lines sl ON sl.sku_id = s.id
    LEFT JOIN shipments sh ON sl.shipment_id = sh.id
    LEFT JOIN retail_locations rl ON sh.destination_id = rl.id AND sh.route_type LIKE '%_to_store'
    WHERE b.batch_number = 'B-001-000021'
    ORDER BY sh.shipment_number
""")

## Q78

We received a customer complaint about shipment SHIP-PLANT-001-000001. Trace backward to the original ingredients and suppliers: which batches were on that shipment, which formulas and ingredients went into those batches, and which suppliers provided the materials? Surface any gaps from missing FK references.

In [ ]:
run_sql(conn, """
    SELECT sh.shipment_number,
           sl.sku_id, s.sku_code,
           b.batch_number, b.production_date,
           f.formula_code,
           i.ingredient_code, i.name as ingredient_name,
           bting.quantity_kg as consumed_kg,
           sup.supplier_code, sup.name as supplier_name
    FROM shipments sh
    LEFT JOIN shipment_lines sl ON sl.shipment_id = sh.id
    LEFT JOIN skus s ON sl.sku_id = s.id
    LEFT JOIN batches b ON b.product_id = s.id AND b.product_type = 'finished_good'
    LEFT JOIN formulas f ON b.formula_id = f.id
    LEFT JOIN batch_ingredients bting ON bting.batch_id = b.id
    LEFT JOIN ingredients i ON bting.ingredient_id = i.id
    LEFT JOIN supplier_ingredients si ON si.ingredient_id = i.id
    LEFT JOIN suppliers sup ON si.supplier_id = sup.id
    WHERE sh.shipment_number = 'SHIP-PLANT-001-000001'
    ORDER BY b.batch_number, i.ingredient_code
""")

## Q79

Goods receipt GR-SHP-1-SUP-001-PLANT-CA-16752 brought in a batch of raw materials. Find every production batch that consumed ingredients from that receipt. I need the lot genealogy for a potential quality hold.

In [ ]:
run_sql(conn, """
    SELECT gr.gr_number,
           grl.ingredient_id, i.ingredient_code, i.name as ingredient_name,
           grl.quantity_kg as received_kg,
           bting.batch_id, b.batch_number, b.production_date,
           bting.quantity_kg as consumed_kg
    FROM goods_receipts gr
    JOIN goods_receipt_lines grl ON grl.gr_id = gr.id
    JOIN ingredients i ON grl.ingredient_id = i.id
    LEFT JOIN batch_ingredients bting ON bting.ingredient_id = grl.ingredient_id
    LEFT JOIN batches b ON bting.batch_id = b.id
    WHERE gr.gr_number = 'GR-SHP-1-SUP-001-PLANT-CA-16752'
    ORDER BY i.ingredient_code, b.batch_number
    LIMIT 50
""")

## Q80

Supplier SUP-019 (Cascade Packaging Inc) just reported a contamination in their facility. Which of our batches used their ingredients, which SKUs did those batches produce, which orders contain those SKUs, and what is the total revenue exposure in dollar terms? Give me the full blast radius.

In [ ]:
# Step 1: Affected ingredients
ingredients_df = run_sql(conn, """
    SELECT DISTINCT i.id, i.ingredient_code, i.name
    FROM suppliers s
    JOIN supplier_ingredients si ON si.supplier_id = s.id
    JOIN ingredients i ON si.ingredient_id = i.id
    WHERE s.supplier_code = 'SUP-019'
""")
print(f"Step 1 \u2014 Ingredients from SUP-019: {len(ingredients_df)}")
display(ingredients_df)

# Step 2: Batches consuming those ingredients
ids = list(ingredients_df['id'])
ph = ','.join(['%s'] * len(ids))
batches = run_sql(conn, f"""
    SELECT DISTINCT b.id, b.batch_number, b.product_type, b.product_id,
           s.sku_code
    FROM batch_ingredients bting
    JOIN batches b ON bting.batch_id = b.id
    LEFT JOIN skus s ON b.product_id = s.id AND b.product_type = 'finished_good'
    WHERE bting.ingredient_id IN ({ph})
""", ids)
print(f"\nStep 2 \u2014 Affected batches: {len(batches)}")
display(batches.head(20))

# Step 3: Orders with those SKUs
sku_ids = list(batches['product_id'].dropna().unique())
if sku_ids:
    ph2 = ','.join(['%s'] * len(sku_ids))
    orders = run_sql(conn, f"""
        SELECT COUNT(DISTINCT o.id) as order_count,
               SUM(ol.quantity_cases) as total_cases
        FROM order_lines ol
        JOIN orders o ON ol.order_id = o.id
        WHERE ol.sku_id IN ({ph2})
    """, sku_ids)
    print(f"\nStep 3 \u2014 Affected orders:")
    display(orders)

    # Step 4: Revenue exposure
    revenue = run_sql(conn, f"""
        SELECT SUM(arl.line_amount) as total_revenue_exposure
        FROM ar_invoice_lines arl
        WHERE arl.sku_id IN ({ph2})
    """, sku_ids)
    print(f"\nStep 4 \u2014 Revenue exposure: ${revenue.iloc[0,0]:,.2f}")

## Q81

Pull the complete procurement document chain for PO PO-CONS-001-001: the purchase order, the goods receipt(s) that received the material, the AP invoice(s) billed against it, and the payment(s) that settled those invoices. Show me the full procure-to-pay paper trail.

In [ ]:
# Purchase Order
po = run_sql(conn, """
    SELECT po.po_number, po.status as po_status,
           s.supplier_code, p.plant_code,
           SUM(pol.quantity_kg * pol.unit_cost) as po_value
    FROM purchase_orders po
    JOIN purchase_order_lines pol ON pol.po_id = po.id
    JOIN suppliers s ON po.supplier_id = s.id
    JOIN plants p ON po.plant_id = p.id
    WHERE po.po_number = 'PO-CONS-001-001'
    GROUP BY po.po_number, po.status, s.supplier_code, p.plant_code
""")
print("Purchase Order:")
display(po)

# Goods Receipts for same supplier/plant
gr = run_sql(conn, """
    SELECT gr.gr_number, gr.status as gr_status, gr.receipt_date,
           SUM(grl.quantity_kg) as received_kg
    FROM goods_receipts gr
    JOIN goods_receipt_lines grl ON grl.gr_id = gr.id
    JOIN purchase_orders po ON gr.plant_id = po.plant_id
    WHERE po.po_number = 'PO-CONS-001-001'
    GROUP BY gr.gr_number, gr.status, gr.receipt_date
    ORDER BY gr.receipt_date
    LIMIT 10
""")
print("\nGoods Receipts:")
display(gr)

# AP Invoices for the supplier
api = run_sql(conn, """
    SELECT api.invoice_number, api.status as inv_status,
           api.invoice_date, api.total_amount
    FROM ap_invoices api
    JOIN purchase_orders po ON api.supplier_id = po.supplier_id
    WHERE po.po_number = 'PO-CONS-001-001'
    ORDER BY api.invoice_date
    LIMIT 10
""")
print("\nAP Invoices:")
display(api)

# Payments
payments = run_sql(conn, """
    SELECT p.payment_date, p.amount, p.net_amount, p.discount_amount,
           api.invoice_number
    FROM ap_payments p
    JOIN ap_invoices api ON p.invoice_id = api.id
    JOIN purchase_orders po ON api.supplier_id = po.supplier_id
    WHERE po.po_number = 'PO-CONS-001-001'
    ORDER BY p.payment_date
    LIMIT 10
""")
print("\nPayments:")
display(payments)

## Q82

Trace the full order-to-cash chain for order ORD-1-CLUB-DC-001-1: the order, the shipment(s) that fulfilled it, the AR invoice(s) generated, and the receipt(s) collected. Flag any AR invoices that are still open or in disputed status — I want to see our collection exposure.

In [ ]:
# Order
order = run_sql(conn, """
    SELECT o.order_number, o.status, o.day, o.total_cases
    FROM orders o WHERE o.order_number = 'ORD-1-CLUB-DC-001-1'
""")
print("Order:")
display(order)

# Shipments
shipments = run_sql(conn, """
    SELECT sh.shipment_number, sh.status, sh.ship_date, sh.arrival_date, sh.freight_cost
    FROM shipments sh
    WHERE sh.order_id = (SELECT id FROM orders WHERE order_number = 'ORD-1-CLUB-DC-001-1')
""")
print("\nShipments:")
display(shipments)

# AR Invoices (linked via shipment)
ar = run_sql(conn, """
    SELECT ari.invoice_number, ari.status, ari.invoice_date,
           ari.total_amount, ari.channel,
           CASE WHEN ari.status IN ('open', 'disputed', 'partial')
                THEN 'ATTENTION' ELSE '' END as flag
    FROM ar_invoices ari
    JOIN shipments sh ON ari.shipment_id = sh.id
    WHERE sh.order_id = (SELECT id FROM orders WHERE order_number = 'ORD-1-CLUB-DC-001-1')
""")
print("\nAR Invoices:")
display(ar)

# AR Receipts
receipts = run_sql(conn, """
    SELECT arr.receipt_date, arr.amount,
           ari.invoice_number
    FROM ar_receipts arr
    JOIN ar_invoices ari ON arr.invoice_id = ari.id
    JOIN shipments sh ON ari.shipment_id = sh.id
    WHERE sh.order_id = (SELECT id FROM orders WHERE order_number = 'ORD-1-CLUB-DC-001-1')
""")
print("\nAR Receipts:")
display(receipts)

## Q83

For every SKU in the Personal Wash category, reconcile total AP spend (what we paid suppliers for ingredients through the BOM) against total AR revenue (what customers paid us). Show the per-SKU margin, and flag any SKUs where invoiced ingredient costs have unexplained variances or where AR invoices are in disputed status.

In [ ]:
# AR revenue per Personal Wash SKU
revenue = run_sql(conn, """
    SELECT s.sku_code, s.name as sku_name,
           SUM(arl.line_amount) as ar_revenue,
           COUNT(DISTINCT ari.id) as invoice_count,
           COUNT(DISTINCT ari.id) FILTER (WHERE ari.status = 'disputed') as disputed_count
    FROM skus s
    JOIN ar_invoice_lines arl ON arl.sku_id = s.id
    JOIN ar_invoices ari ON arl.invoice_id = ari.id
    WHERE s.category = 'Personal Wash'
    GROUP BY s.sku_code, s.name
""")

# BOM cost per SKU
pw_skus = run_sql(conn, "SELECT sku_code FROM skus WHERE category = 'Personal Wash' AND is_active = true")
costs = []
for sku_code in pw_skus['sku_code']:
    bom = explode_bom(conn, sku_code=sku_code, resolve_costs=True)
    if bom:
        total_cost = sum(r['cumulative_quantity_kg'] * (r.get('cheapest_unit_cost') or 0) for r in bom)
        costs.append({'sku_code': sku_code, 'bom_cost': total_cost})
cost_df = pd.DataFrame(costs)

# Merge and calculate margin
merged = revenue.merge(cost_df, on='sku_code', how='outer')
merged['margin'] = merged['ar_revenue'].fillna(0) - merged['bom_cost'].fillna(0)
merged['flag'] = ''
merged.loc[merged['disputed_count'] > 0, 'flag'] = 'DISPUTED AR'
merged = merged.sort_values('margin', ascending=False)
display(merged)

## Q84

A return was filed under RMA-001-0-1069. Trace the returned SKU back through its production history: which batch produced it, which formula was used, and which ingredients and suppliers were involved? I need to know if the defect traces to a raw material.

In [ ]:
run_sql(conn, """
    SELECT r.return_number, r.return_date,
           rl_line.sku_id, s.sku_code, rl_line.quantity_cases, rl_line.condition,
           b.batch_number, b.production_date,
           f.formula_code,
           i.ingredient_code, i.name as ingredient_name,
           sup.supplier_code, sup.name as supplier_name
    FROM returns r
    JOIN return_lines rl_line ON rl_line.return_id = r.id
    JOIN skus s ON rl_line.sku_id = s.id
    LEFT JOIN batches b ON b.product_id = s.id AND b.product_type = 'finished_good'
    LEFT JOIN formulas f ON b.formula_id = f.id
    LEFT JOIN batch_ingredients bting ON bting.batch_id = b.id
    LEFT JOIN ingredients i ON bting.ingredient_id = i.id
    LEFT JOIN supplier_ingredients si ON si.ingredient_id = i.id
    LEFT JOIN suppliers sup ON si.supplier_id = sup.id
    WHERE r.return_number = 'RMA-001-0-1069'
    ORDER BY rl_line.sku_id, b.batch_number, i.ingredient_code
    LIMIT 50
""")

## Q85

What is the cheapest possible path to get SKU-ORAL-001 from raw materials to retail location STORE-RET-001-0042? Combine the BOM ingredient costs, the cheapest supplier for each ingredient, the lowest-cost inbound transport to PLANT-TX, the production cost, and the cheapest outbound route to the retail location. Give me the total raw-to-shelf unit cost.

In [ ]:
G = build_transport_graph(conn)
plant_tx = resolve_location_key(conn, 'PLANT-TX')
store = resolve_location_key(conn, 'STORE-RET-001-0042')

# Step 1: BOM with cheapest supplier costs
bom = explode_bom(conn, sku_code='SKU-ORAL-001', resolve_costs=True)
material_cost = sum(r['cumulative_quantity_kg'] * (r.get('cheapest_unit_cost') or 0) for r in bom)
print(f"Step 1 \u2014 Raw material cost: ${material_cost:.2f}")

# Step 2: Inbound transport (supplier -> PLANT-TX)
inbound_km = 0
suppliers_seen = set()
for r in bom:
    sup_code = r.get('cheapest_supplier_code')
    if sup_code and sup_code not in suppliers_seen:
        suppliers_seen.add(sup_code)
        sup_id = run_sql(conn, "SELECT id FROM suppliers WHERE supplier_code = %s", (sup_code,))
        if len(sup_id) > 0:
            sup_key = f"supplier:{sup_id.iloc[0,0]}"
            try:
                dist = nx.shortest_path_length(G, sup_key, plant_tx, weight='distance_km')
                inbound_km += dist
            except nx.NetworkXNoPath:
                pass
inbound_freight = inbound_km * 0.05  # illustrative rate
print(f"Step 2 \u2014 Inbound freight: {inbound_km:.0f} km, est cost: ${inbound_freight:.2f}")

# Step 3: Production cost (formula metadata)
prod = run_sql(conn, """
    SELECT f.batch_size_kg, f.yield_percent
    FROM formulas f
    JOIN skus s ON f.product_id = s.id AND f.bom_level = 0
    WHERE s.sku_code = 'SKU-ORAL-001'
""")
print(f"Step 3 \u2014 Production: batch_size={prod.iloc[0,0]} kg, yield={prod.iloc[0,1]}%")

# Step 4: Outbound transport (PLANT-TX -> store)
try:
    outbound_dist = nx.shortest_path_length(G, plant_tx, store, weight='distance_km')
    outbound_path = nx.shortest_path(G, plant_tx, store, weight='distance_km')
    outbound_freight = outbound_dist * 0.05
    print(f"Step 4 \u2014 Outbound freight: {outbound_dist:.0f} km, est cost: ${outbound_freight:.2f}")
    display(display_path(G, outbound_path, 'distance_km'))
except nx.NetworkXNoPath:
    outbound_freight = 0
    print("Step 4 \u2014 No outbound path found")

# Total
total = material_cost + inbound_freight + outbound_freight
print(f"\n{'='*50}")
print(f"Total raw-to-shelf cost per batch:")
print(f"  Material:         ${material_cost:.2f}")
print(f"  Inbound freight:  ${inbound_freight:.2f}")
print(f"  Outbound freight: ${outbound_freight:.2f}")
print(f"  TOTAL:            ${total:.2f}")

In [ ]:
conn.close()
print("Session closed.")